In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!git clone https://github.com/yjs1224/TextSteganalysis.git

Cloning into 'TextSteganalysis'...
remote: Enumerating objects: 136, done.
remote: Counting objects: 100% (136/136), done.
remote: Compressing objects: 100% (104/104), done.
remote: Total 136 (delta 70), reused 67 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (136/136), 67.93 KiB | 539.00 KiB/s, done.
Resolving deltas: 100% (70/70), done.


In [3]:
%cd TextSteganalysis/

/content/TextSteganalysis


In [4]:
!pip install stanza

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 586.9/586.9 kB 1.8 MB/s eta 0:00:00


In [5]:
BASE_PATH = "/content/drive/MyDrive/Colab/steganalysis"
DATA_PATH = f"{BASE_PATH}/data"

TRAIN_DATASET_PATH = f"{DATA_PATH}/steganalysis_dataset_2.csv"
TEST_DATASET_PATH = f"{DATA_PATH}/synsteg_dataset.zip"

# TextSteganalysis

In [6]:
import os
import json

import pandas as pd

### Train Config Definition

In [14]:
config = {
  "use_processor": True,
  "out_dir": "out_dir",
  "checkpoint": "best",
  "record_file": "record.txt",
  "model": "sesy",
  "class_num": 2,
  "tokenizer": False,
  "seed": 42,
  "gpuid": "0",
  "task_name": "steganalysis",
  "use_plm": True,
  "repeat_num": 1,
  "device": "cuda:0",
  "Dataset": {
    "name": "tweet",
    "stego_file": f"{DATA_PATH}/synsteg_stego.txt",
    "cover_file": f"{DATA_PATH}/synsteg_clean.txt",
    "csv_dir": DATA_PATH,
    "resplit": True,
    "split_ratio": 0.8,
    "save_cache": False,
    "overwrite_cache": True
  },
  "Tokenizer": {
    "model_name_or_path": "bert-base-uncased"
  },
  "Training_with_Processor": {
    "num_train_epochs": 3,
    "learning_rate": 5e-5,
    "model_name_or_path": "bert-base-uncased",
    "do_lower_case":True,
    "per_gpu_train_batch_size": 32,
    "per_gpu_eval_batch_size": 32,
    "n_gpu": 1,
    "max_steps": -1,
    "gradient_accumulation_steps": 1,
    "eval_and_save_steps": 100,
    "warmup_ratio": 0.06,
    "weight_decay": 0.01,
    "adam_epsilon": 1e-8,
    "max_grad_norm": 1.0,
    "logging_steps": -1,
    "evaluate_during_training": False,
    "save_only_best": True,
    "use_fixed_seq_length": True,
    "eval_all_checkpoints": False,
    "skip_evaluate_dev":True
  },
  "Training": {
    "batch_size": 100,
    "epoch": 10,
    "learning_rate": 0.001,
    "early_stop": 50,
    "warmup_ratio": 0.06,
    "weight_decay": 0.01,
    "adam_epsilon": 1e-8,
    "evaluate_during_training": False,
    "save_only_best": True
  },
  "Vocabulary": {
    "word_drop": 0,
    "do_lower": True,
    "max_length": 60
  },
  "CNN": {
	"embed_size": 128,
	"filter_num": 128,
	"filter_size": [3, 4, 5],
    "dropout_rate": 0.2,
    "criteration": "CrossEntropyLoss"
  },
  "RNN": {
    "cell":"bi-lstm",
    "embed_size": 128,
    "hidden_dim": 256,
	"num_layers": 1,
    "dropout_rate": 0.2,
    "criteration": "CrossEntropyLoss"
  },
  "FCN": {
    "embed_size": 128,
    "dropout_rate": 0.2,
    "criteration": "CrossEntropyLoss"
  },
  "LSTMATT": {
    "embed_size":  128,
    "hidden_dim": 256,
    "dropout_rate":0.2,
    "criteration": "CrossEntropyLoss",
    "bidirectional": True
  },
  "RBiLSTMC": {
    "num_layers": 1,
    "kernel_sizes": [3,4,5],
    "kernel_num": 128,
    "embed_dim": 128,
    "hidden_dim": 256,
    "LSTM_dropout": 0.2,
    "CNN_dropout": 0.2,
    "Ci": 1,
    "criteration": "CrossEntropyLoss"
  },
  "BiLSTMDENSE": {
    "num_layers": 1,
    "embed_dim": 256,
    "hidden_dim": 200,
    "dropout_rate": 0.2,
    "criteration": "CrossEntropyLoss"
  },
  "SESY": {
    "clf": "cnn",
    "criteration": "CrossEntropyLoss",
    "strategy": "cas",
    "embed_size": 100,
    "hidden_dim": 128,
    "readout_size": 64,
    "gat_alpha": 0.2,
    "gat_heads": 8,
    "dropout_rate": 0.2,
    "TC_configs": {
      "cnn": {
        "filter_num": 128,
        "filter_size": [3, 4, 5]

      },
      "rnn": {
        "cell":"bi-lstm",
        "hidden_dim": 256,
        "num_layers": 1
      },
      "fc": {
      }
    }
  }
}

In [15]:
with open('config.json', 'w') as f:
    json.dump(config, f)

### Prepare Training Data

In [ ]:
train_df = pd.read_csv(TRAIN_DATASET_PATH)
train_df.head()

,dataset,label,model,coding,text
0,movie,neg,NaN,NaN,i always appreciate movies taking place in new...
1,movie,neg,NaN,NaN,it 's an inner look into the darkest parts of ...
2,movie,neg,NaN,NaN,the subtle brilliance in the film lies in the ...
3,movie,neg,NaN,NaN,who wants to see teenager kay lenz being bruta...
4,movie,neg,NaN,NaN,warfield lends a quiet maturity to the role of...


In [ ]:
with open(f"{DATA_PATH}/stego.txt", "w") as f:
    f.write('\n'.join(train_df[train_df["label"] == "pos"].text.values.tolist()))


with open(f"{DATA_PATH}/cover.txt", "w") as f:
    f.write('\n'.join(train_df[train_df["label"] == "neg"].text.values.tolist()))

### Run Training

In [17]:
for model_name in ["rbilstmc", "lstmatt", "cnn", "bilstmdense", "birnn"]:  # "sesy"
    config["model"] = model_name
    config["out_dir"] = f"{BASE_PATH}/synsteg_out_dir_{model_name}_synsteg_test"

    if model_name in ["sesy", "lstmatt"]:
        config["use_plm"] = True
    else:
        config["use_plm"] = False

    if model_name == "sesy":
        config["task_name"] = "graph_steganalysis"


    with open('config.json', 'w') as f:
        json.dump(config, f)

    !python main.py --config_path config.json

2024-11-11 10:34:00.889584: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-11-11 10:34:00.908424: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-11-11 10:34:00.914107: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-11-11 10:34:02.050111: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
11/11/2024 10:34:02 - INFO - __main__ -   --------------Main Configs-------------------
11/11/2024 10:34:02 - INFO - __main__ -   {'use_processor': True, 'out_dir': '/content/drive/MyDrive/Colab/steganalysis/synsteg_out_dir_rbilstmc_synsteg_

In [18]:
from transformers import AutoTokenizer
from main import load_model, evaluate, load_and_cache_examples
import utils

TASK_NAME = "steganalysis"
SPLIT = "test"
PREFIX = "best"

checkpoints = [
    # "/content/drive/MyDrive/Colab/steganalysis/synsteg_out_dir_ts-csw_synsteg/best",
    "/content/drive/MyDrive/Colab/steganalysis/synsteg_out_dir_rbilstmc_synsteg/best",
    "/content/drive/MyDrive/Colab/steganalysis/synsteg_out_dir_lstmatt_synsteg/best",
    "/content/drive/MyDrive/Colab/steganalysis/synsteg_out_dir_cnn_synsteg/best",
    "/content/drive/MyDrive/Colab/steganalysis/synsteg_out_dir_bilstmdense_synsteg/best",
    "/content/drive/MyDrive/Colab/steganalysis/synsteg_out_dir_birnn_synsteg/best"
]

results = {}
for checkpoint in checkpoints:
    # If using additional prefix change last index to -3
    model_name = checkpoint.split('/')[-2].split('_')[-2]
    config["model"] = model_name
    config["out_dir"] = "/".join(checkpoint.split("/")[:-1])
    if model_name in ["lstmatt"]:
        config["use_plm"] = True
    else:
        config["use_plm"] = False

    with open('config.json', 'w') as f:
        json.dump(config, f)

    Configs = utils.Config("config.json").get_configs()

    tokenizer = AutoTokenizer.from_pretrained(checkpoint, do_lower_case=True)
    print(checkpoint)
    model = load_model(Configs, VOCAB_SIZE=tokenizer.vocab_size, checkpoint=checkpoint)

    eval_dataset = load_and_cache_examples(Configs.Dataset, TASK_NAME, tokenizer, split=SPLIT)
    true_labels = [eval_dataset[idx][-1].numpy().tolist() for idx in range(len(eval_dataset))]

    result, preds, ex_ids = evaluate(model, tokenizer, Configs, TASK_NAME, split=SPLIT, prefix=PREFIX)

    results[checkpoint] = {
        "result": result,
        "preds": preds,
        "ex_idx": ex_ids,
        "true_labels": true_labels,
        "confusion_matrix": {
            "tp": sum([p == t for p, t in zip(preds, true_labels) if t == 1]),
            "fp": sum([p != t for p, t in zip(preds, true_labels) if t == 0]),
            "tn": sum([p == t for p, t in zip(preds, true_labels) if t == 0]),
            "fn": sum([p != t for p, t in zip(preds, true_labels) if t == 1])
        }
    }

/content/drive/MyDrive/Colab/steganalysis/synsteg_out_dir_rbilstmc_synsteg/best


/usr/local/lib/python3.10/dist-packages/torch/nn/modules/rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  warnings.warn(
Evaluating: 100%|██████████| 55/55 [00:01<00:00, 49.21it/s]


/content/drive/MyDrive/Colab/steganalysis/synsteg_out_dir_lstmatt_synsteg/best


Evaluating: 100%|██████████| 55/55 [00:16<00:00,  3.30it/s]
/content/TextSteganalysis/main.py:90: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model = torch.load(os.path.jo

/content/drive/MyDrive/Colab/steganalysis/synsteg_out_dir_cnn_synsteg/best


Evaluating: 100%|██████████| 55/55 [00:00<00:00, 158.23it/s]
/usr/local/lib/python3.10/dist-packages/torch/nn/modules/rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  warnings.warn(
/content/TextSteganalysis/main.py:90: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.

/content/drive/MyDrive/Colab/steganalysis/synsteg_out_dir_bilstmdense_synsteg/best


Evaluating: 100%|██████████| 55/55 [00:01<00:00, 34.45it/s]
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/torch/nn/modules/rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  warnings.warn(
/content/TextSteganalysis/main.py:90: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more de

/content/drive/MyDrive/Colab/steganalysis/synsteg_out_dir_birnn_synsteg/best


Evaluating: 100%|██████████| 55/55 [00:00<00:00, 77.82it/s]


In [19]:
for checkpoint in checkpoints:
    cm = results[checkpoint]["confusion_matrix"]
    accuracy = (cm['tp'] + cm['tn']) / (cm['tp'] + cm['tn'] + cm['fn'] + cm['fp'])
    if not (cm['tp'] + cm['fp']):
        precision = 0.0
    else:
        precision = cm['tp'] / (cm['tp'] + cm['fp'])

    recall = cm['tp'] / (cm['tp'] + cm['fn'])
    if not precision + recall:
        f1_score = 0
    else:
        f1_score = (2 * precision * recall) / (precision + recall)

    # If using additional prefix change last index to -3
    model_name = checkpoint.split('/')[-2].split('_')[-2]

    print(model_name.upper())
    print(cm)
    print(f"accuracy = {accuracy:.4f}")
    print(f"precision = {precision:.4f}")
    print(f"recal = {recall:.4f}")
    print(f"f1 score = {f1_score:.4f}")

RBILSTMC
{'tp': 438, 'fp': 86, 'tn': 785, 'fn': 424}
accuracy = 0.7057
precision = 0.8359
recal = 0.5081
f1 score = 0.6320
LSTMATT
{'tp': 462, 'fp': 9, 'tn': 862, 'fn': 400}
accuracy = 0.7640
precision = 0.9809
recal = 0.5360
f1 score = 0.6932
CNN
{'tp': 452, 'fp': 95, 'tn': 776, 'fn': 410}
accuracy = 0.7086
precision = 0.8263
recal = 0.5244
f1 score = 0.6416
BILSTMDENSE
{'tp': 0, 'fp': 0, 'tn': 871, 'fn': 862}
accuracy = 0.5026
precision = 0.0000
recal = 0.0000
f1 score = 0.0000
BIRNN
{'tp': 460, 'fp': 191, 'tn': 680, 'fn': 402}
accuracy = 0.6578
precision = 0.7066
recal = 0.5336
f1 score = 0.6081


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
model_to_embed = {}

In [ ]:
# TSNE Visualization

from glob import glob
from transformers import AutoTokenizer

from main import load_model, evaluate, load_and_cache_examples
import utils
import torch

TASK_NAME = "steganalysis"
SPLIT = "test"
PREFIX = "best"

checkpoints = [ckpt + '/best' for ckpt in glob(f'{BASE_PATH}/synsteg_*_test')]
print('Found', len(checkpoints), 'checkpoints')

for checkpoint in checkpoints:
    print(checkpoint)
    model_name = checkpoint.split('/')[-2].split('_')[-3]
    if model_name in model_to_embed:
        continue

    print('Model name:', model_name)

    config["model"] = model_name
    config["out_dir"] = f"{BASE_PATH}/synsteg_out_dir_{model_name}_synsteg_test"
    if model_name in ["lstmatt"]:
        config["use_plm"] = True
    else:
        config["use_plm"] = False

    with open('config.json', 'w') as f:
        json.dump(config, f)

    Configs = utils.Config("config.json").get_configs()

    tokenizer = AutoTokenizer.from_pretrained(checkpoint, do_lower_case=True)
    model = load_model(Configs, VOCAB_SIZE=tokenizer.vocab_size, checkpoint=checkpoint)

    eval_dataset = load_and_cache_examples(Configs.Dataset, TASK_NAME, tokenizer, split=SPLIT)
    true_labels = [eval_dataset[idx][-1].numpy().tolist() for idx in range(len(eval_dataset))]

    _, _, _, embeddings = evaluate(model, tokenizer, Configs, TASK_NAME, split=SPLIT, prefix=PREFIX)
    print(embeddings)
    combined_embedding = embeddings[0]
    for embedding in embeddings[1:]:
        combined_embedding = torch.cat((combined_embedding, embedding), 0)

    model_to_embed[model_name] = torch.flatten(combined_embedding, start_dim=1).detach().cpu().numpy()


In [ ]:
from sklearn.manifold import TSNE
import plotly.express as px
import matplotlib.pyplot as plt

model_to_tsne = {}

for model_name, embeddings in model_to_embed.items():
    if embeddings is None:
        continue

    embed = TSNE(
        n_components=2,
        perplexity=10,
        early_exaggeration=12,
        learning_rate=200,
        n_iter=5000,
        n_iter_without_progress=300,
        min_grad_norm=0.0000001,
        metric='euclidean',
        init='random',
        verbose=0,
        random_state=42,
        method='barnes_hut',
        angle=0.5,
        n_jobs=-1
    )


    tsne_embedded = embed.fit_transform(embeddings)
    print(tsne_embedded.shape)
    model_to_tsne[model_name] = tsne_embedded

/usr/local/lib/python3.10/dist-packages/sklearn/manifold/_t_sne.py:1162: FutureWarning: 'n_iter' was renamed to 'max_iter' in version 1.5 and will be removed in 1.7.
  warnings.warn(


(493, 2)


/usr/local/lib/python3.10/dist-packages/sklearn/manifold/_t_sne.py:1162: FutureWarning: 'n_iter' was renamed to 'max_iter' in version 1.5 and will be removed in 1.7.
  warnings.warn(


(493, 2)


/usr/local/lib/python3.10/dist-packages/sklearn/manifold/_t_sne.py:1162: FutureWarning: 'n_iter' was renamed to 'max_iter' in version 1.5 and will be removed in 1.7.
  warnings.warn(


...

(493, 2)


/usr/local/lib/python3.10/dist-packages/sklearn/manifold/_t_sne.py:1162: FutureWarning: 'n_iter' was renamed to 'max_iter' in version 1.5 and will be removed in 1.7.
  warnings.warn(


(493, 2)


/usr/local/lib/python3.10/dist-packages/sklearn/manifold/_t_sne.py:1162: FutureWarning: 'n_iter' was renamed to 'max_iter' in version 1.5 and will be removed in 1.7.
  warnings.warn(


(493, 2)


In [ ]:
import plotly.graph_objects as go

fig.add_trace(go.Scatter(x=[1, 2, 3], y=[4, 5, 6]),
              row=1, col=1)

fig.add_trace(go.Scatter(x=[20, 30, 40], y=[50, 60, 70]),
              row=1, col=2)

fig.add_trace(go.Scatter(x=[300, 400, 500], y=[600, 700, 800]),
              row=2, col=1)

fig.add_trace(go.Scatter(x=[4000, 5000, 6000], y=[7000, 8000, 9000]),
              row=2, col=2)

fig.show()


In [ ]:
for model_name, tsne_embedding in model_to_tsne.items():
    fig = px.scatter(None, x=tsne_embedding[:,0], y=tsne_embedding[:,1],
                    labels={"x": "Dimension 1", "y": "Dimension 2"},
                    opacity=1, color=['stego' if i == 1 else 'cover' for i in true_labels],
                    width=800, height=800)

    fig.update_layout(dict(plot_bgcolor = 'white'))

    fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='lightgrey',
                    zeroline=True, zerolinewidth=1, zerolinecolor='lightgrey',
                    showline=True, linewidth=1, linecolor='black')

    fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='lightgrey',
                    zeroline=True, zerolinewidth=1, zerolinecolor='lightgrey',
                    showline=True, linewidth=1, linecolor='black')

    fig.update_layout(title_text=f"t-SNE for embeddings from {model_name.upper()}")
    fig.show()

# Text-Steganalysis

In [ ]:
!git clone https://github.com/CAU-Tstega/Text-steganalysis.git

Cloning into 'Text-steganalysis'...
remote: Enumerating objects: 199, done.
remote: Total 199 (delta 0), reused 0 (delta 0), pack-reused 199
Receiving objects: 100% (199/199), 51.25 KiB | 12.81 MiB/s, done.
Resolving deltas: 100% (93/93), done.


In [ ]:
%cd Text-steganalysis

/content/Text-steganalysis


In [ ]:
!pip install torchtext==0.6.0 -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.2/64.2 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 71.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 68.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 99.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 8.3 MB/s eta 0:00:00
ERROR: Operation cancelled by user


In [ ]:
save_dir = f"{BASE_PATH}/BiLSTM-Dense-trained"

In [ ]:
args = f"-batch-size 16 -epochs 10 -test-interval 200 -save-interval 200 -save-dir {save_dir} "
args += f"-test-cover-dir {DATA_PATH}/synsteg_clean_test.txt -test-stego-dir {DATA_PATH}/synsteg_stego_test.txt "
args+= f"-train-cover-dir {DATA_PATH}/synsteg_clean_train.txt -train-stego-dir {DATA_PATH}/synsteg_stego_train.txt"

In [ ]:
!python3 BiLISTM-Dense/main.py {args}

In [ ]:
args += " -test y"

!python3 BiLISTM-Dense/main.py {args}


Loading data...
loading the training data...
loading the testing data...

----------testing------------
Loading test model from /content/drive/MyDrive/Colab/steganalysis/BiLSTM-Dense-trained...
the /content/drive/MyDrive/Colab/steganalysis/BiLSTM-Dense-trained/best_steps_1200.pt model is loaded...

Testing - loss:0.697300 acc:0.5000(463/926)


In [ ]:
!wget "https://nlp.stanford.edu/data/glove.6B.zip"

In [ ]:
!unzip glove.6B.zip

Archive:  glove.6B.zip
  inflating: glove.6B.50d.txt        
  inflating: glove.6B.100d.txt       
  inflating: glove.6B.200d.txt       
  inflating: glove.6B.300d.txt       


In [ ]:
save_dir = f"{BASE_PATH}/LS_CNN-trained"

In [ ]:
%cd LS_CNN

[Errno 2] No such file or directory: 'LS_CNN'
/content/Text-steganalysis/LS_CNN


In [ ]:
args = f"-batch-size 16 -epochs 20 -test-interval 200 -save-interval 200 -save-dir {save_dir} "
args += f"-test-cover-dir {DATA_PATH}/synsteg_clean_test.txt -test-stego-dir {DATA_PATH}/synsteg_stego_test.txt "
args+= f"-train-cover-dir {DATA_PATH}/synsteg_clean_train.txt -train-stego-dir {DATA_PATH}/synsteg_stego_train.txt"

In [ ]:
!python3 main.py {args}

In [ ]:
args += " -test y"
!python3 main.py {args}


Loading data...
loading the training data...
loading the testing data...

----------testing------------
Loading test model from /content/drive/MyDrive/Colab/steganalysis/LS_CNN-trained...
the /content/drive/MyDrive/Colab/steganalysis/LS_CNN-trained/best_steps_2400.pt model is loaded...

Testing - loss:0.691627 acc:0.5000(463/926)
the /content/drive/MyDrive/Colab/steganalysis/LS_CNN-trained/best_steps_3600.pt model is loaded...

Testing - loss:0.697983 acc:0.5000(463/926)
the /content/drive/MyDrive/Colab/steganalysis/LS_CNN-trained/best_steps_6800.pt model is loaded...

Testing - loss:0.693658 acc:0.5000(463/926)


In [ ]:
%cd ../MS-TL/Fine-tuning-BERT/

/content/Text-steganalysis/MS-TL/Fine-tuning-BERT


In [ ]:
save_dir = f"{BASE_PATH}/MS-TL-trained"

In [ ]:
args = f"-batch-size 16 -epochs 10 -test-interval 200 -save-interval 200 -save-dir {save_dir} "
args += f"-test-cover-dir {DATA_PATH}/synsteg_clean_test.txt -test-stego-dir {DATA_PATH}/synsteg_stego_test.txt "
args+= f"-train-cover-dir {DATA_PATH}/synsteg_clean_train.txt -train-stego-dir {DATA_PATH}/synsteg_stego_train.txt"

In [ ]:
!python3 main.py {args}


Loading data...
100% 4162/4162 [00:01<00:00, 2249.06it/s]
100% 4162/4162 [00:02<00:00, 1740.71it/s]
100% 463/463 [00:00<00:00, 2491.60it/s]
100% 463/463 [00:00<00:00, 2397.33it/s]

--------training epochs: 1-----------
Batch[200] - loss:0.695682 acc:0.4375(7/16)
Validation - loss:0.693122 acc:0.4941(419/848)
Batch[400] - loss:0.696515 acc:0.5000(8/16)
Validation - loss:0.695905 acc:0.4953(420/848)
Batch[460] - loss:0.698167 acc:0.3750(6/16)
--------training epochs: 2-----------
Batch[600] - loss:0.693022 acc:0.5000(8/16)
Validation - loss:0.694022 acc:0.4953(420/848)
Batch[800] - loss:0.693498 acc:0.5000(8/16)
Validation - loss:0.693157 acc:0.4870(413/848)
Batch[920] - loss:0.686100 acc:0.5625(9/16)
--------training epochs: 3-----------
Batch[1000] - loss:0.683577 acc:0.6250(10/16)
Validation - loss:0.694963 acc:0.4953(420/848)
Batch[1200] - loss:0.690860 acc:0.5625(9/16)
Validation - loss:0.693014 acc:0.4941(419/848)
Batch[1400] - loss:0.693513 acc:0.3125(5/16)
Validation - loss:0.69

In [ ]:
args += " -test y"
!python3 main.py {args}


Loading data...
100% 4162/4162 [00:04<00:00, 869.38it/s] 
100% 4162/4162 [00:01<00:00, 2337.10it/s]
100% 463/463 [00:00<00:00, 2243.57it/s]
100% 463/463 [00:00<00:00, 2552.15it/s]

----------testing------------
Loading test model from /content/drive/MyDrive/Colab/steganalysis/MS-TL-trained...
the /content/drive/MyDrive/Colab/steganalysis/MS-TL-trained/best_steps_200.pt model is loaded...
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))

Testing - loss:0.713160 acc:0.5000(463/928)
the /content/drive/MyDrive/Colab/steganalysis/MS-TL-trained/best_steps_400.pt model is loaded...

Testing - loss:0.660339 acc:0.5000(463/928)


In [ ]:
%cd ../KD&FI

/content/Text-steganalysis/MS-TL/KD&FI


In [ ]:
save_dir = f"{BASE_PATH}/MS-TL-KD-lstm-trained"

In [ ]:
args = f"-batch-size 16 -epochs 10 -test-interval 200 -save-interval 200 -save-dir {save_dir} -load-dir {BASE_PATH}/MS-TL-trained/best_steps_400.pt "
args += f"-test-cover-dir {DATA_PATH}/synsteg_clean_test.txt -test-stego-dir {DATA_PATH}/synsteg_stego_test.txt "
args+= f"-train-cover-dir {DATA_PATH}/synsteg_clean_train.txt -train-stego-dir {DATA_PATH}/synsteg_stego_train.txt"

In [ ]:
!python3 main.py {args}


Loading data...
100% 4162/4162 [00:01<00:00, 2178.01it/s]
100% 4162/4162 [00:01<00:00, 2189.70it/s]
100% 463/463 [00:00<00:00, 2396.68it/s]
100% 463/463 [00:00<00:00, 2473.07it/s]
Knowledge Distilling...
teacher:  109483778
student:  9960602

--------training epochs: 1-----------
Batch[200] - loss:0.034704 acc:0.5000(8/16)
Validation - loss:0.700239 acc:0.4717(400/848)
Batch[400] - loss:0.036038 acc:0.3750(6/16)
Validation - loss:0.697876 acc:0.4682(397/848)
Batch[460] - loss:0.036867 acc:0.4375(7/16)
--------training epochs: 2-----------
Batch[600] - loss:0.034616 acc:0.6250(10/16)
Validation - loss:0.701473 acc:0.4634(393/848)
Batch[800] - loss:0.035806 acc:0.3750(6/16)
Validation - loss:0.700089 acc:0.4658(395/848)
Batch[920] - loss:0.035716 acc:0.3125(5/16)
--------training epochs: 3-----------
Batch[1000] - loss:0.034954 acc:0.5000(8/16)
Validation - loss:0.700684 acc:0.4729(401/848)
Batch[1200] - loss:0.034843 acc:0.5000(8/16)
Validation - loss:0.700399 acc:0.4646(394/848)
Batch

In [ ]:
args += " -test y"
!python3 main.py {args}


Loading data...
100% 4162/4162 [00:02<00:00, 1786.28it/s]
100% 4162/4162 [00:03<00:00, 1253.55it/s]
100% 463/463 [00:00<00:00, 1303.76it/s]
100% 463/463 [00:00<00:00, 1301.70it/s]
Knowledge Distilling...
teacher:  109483778
student:  9960602

----------testing------------
Loading test model from /content/drive/MyDrive/Colab/steganalysis/MS-TL-KD-lstm-trained...
the /content/drive/MyDrive/Colab/steganalysis/MS-TL-KD-lstm-trained/best_steps_200.pt model is loaded...
Testing - loss:0.682282 acc:0.5000(463/928)

the /content/drive/MyDrive/Colab/steganalysis/MS-TL-KD-lstm-trained/best_steps_1000.pt model is loaded...
Testing - loss:0.686116 acc:0.5000(463/928)

the /content/drive/MyDrive/Colab/steganalysis/MS-TL-KD-lstm-trained/best_steps_1600.pt model is loaded...
Testing - loss:0.692695 acc:0.5000(463/928)



In [ ]:
%cd R-BiLSTM-C/

/content/Text-steganalysis/R-BiLSTM-C


In [ ]:
save_dir = f"{BASE_PATH}/R-BiLSTM-C-trained"

In [ ]:
args = f"-batch-size 16 -epochs 10 -test-interval 200 -save-interval 200 -save-dir {save_dir} "
args += f"-test-cover-dir {DATA_PATH}/synsteg_clean_test.txt -test-stego-dir {DATA_PATH}/synsteg_stego_test.txt "
args+= f"-train-cover-dir {DATA_PATH}/synsteg_clean_train.txt -train-stego-dir {DATA_PATH}/synsteg_stego_train.txt"

In [ ]:
!python3 main.py {args}


Loading data...
loading the training data...

--------training epochs: 1-----------
Batch[200] - loss:0.773229 acc:0.5000(8/16)
Validation - loss:0.707085 acc:0.4586(382/833)
Batch[400] - loss:0.800997 acc:0.3125(5/16)
Validation - loss:0.698981 acc:0.4838(403/833)
Batch[460] - loss:0.677202 acc:0.5000(8/16)
--------training epochs: 2-----------
Batch[600] - loss:0.626021 acc:0.6250(10/16)
Validation - loss:0.756543 acc:0.4934(411/833)
Batch[800] - loss:0.729178 acc:0.4375(7/16)
Validation - loss:0.706913 acc:0.4694(391/833)
Batch[920] - loss:0.789678 acc:0.3125(5/16)
--------training epochs: 3-----------
Batch[1000] - loss:0.741876 acc:0.4375(7/16)
Validation - loss:0.732508 acc:0.5042(420/833)
Batch[1200] - loss:0.747859 acc:0.5000(8/16)
Validation - loss:0.709334 acc:0.4946(412/833)
Batch[1400] - loss:0.725174 acc:0.3750(6/16)
Validation - loss:0.704308 acc:0.4118(343/833)

--------training epochs: 4-----------
Batch[1600] - loss:0.735729 acc:0.4375(7/16)
Validation - loss:0.715268

In [ ]:
args += " -test y"
!python3 main.py {args}


Loading data...
loading the training data...
loading the testing data...

----------testing------------
Loading test model from /content/drive/MyDrive/Colab/steganalysis/R-BiLSTM-C-trained...
the /content/drive/MyDrive/Colab/steganalysis/R-BiLSTM-C-trained/best_steps_400.pt model is loaded...

Testing - loss:0.695917 acc:0.5000(463/926)
the /content/drive/MyDrive/Colab/steganalysis/R-BiLSTM-C-trained/best_steps_600.pt model is loaded...

Testing - loss:0.743852 acc:0.5000(463/926)
the /content/drive/MyDrive/Colab/steganalysis/R-BiLSTM-C-trained/best_steps_1000.pt model is loaded...

Testing - loss:0.720265 acc:0.5000(463/926)
